# Dựng clean/ từ dữ liệu thi — CPU, KHÔNG GPU

Vì sao có notebook này: `legalqa-clean` nằm ở tài khoản `sunny0703`, mà dataset là
riêng từng tài khoản (thử `kaggle datasets files sunny0703/legalqa-clean` bằng token
`tunvln` → 403, trong khi `sunny070306/official` → OK, nên đó là từ chối thật chứ
không phải lỗi endpoint). `sunny0703` đã cạn quota GPU (30,68/30 giờ, refresh
2026-09-19) nên phần còn lại của pipeline phải chạy bằng `tunvln`.

Chuyển 1,24 GB qua máy dev rồi đẩy lên lại là đường đắt nhất. `clean/` dựng lại được
từ `official` — thứ `tunvln` ĐỌC ĐƯỢC — trong ~26 phút **CPU thuần**. Kernel tắt GPU
thì 26 phút đó không đụng vào quota GPU 30 giờ.

Không chạy `build_gold.py`: gold chỉ dùng cho train/eval retrieval, `infer.py` không
đọc tới. Thêm nó là thêm thời gian và thêm chỗ để hỏng, không đổi lại gì cho lượt nộp.

Cần gắn: Accelerator **None**, dataset `official` + `legalqa-v10-src`.


In [ ]:
import os, sys, shutil, zipfile, subprocess
from pathlib import Path

SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if (cand.name == "legalqa" and (cand / "bm25.py").exists()
            and "legalqa-v10-src" in str(cand)):
        SRC = cand.parent.parent          # .../src/legalqa -> gốc repo
        break
if SRC is None:
    z = next(Path("/kaggle/input").glob("**/src.zip"), None)
    assert z, "Không thấy codebase — chưa gắn dataset legalqa-v10-src"
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/unz")
    SRC = Path("/kaggle/working/unz")

REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, "src")
print("codebase:", (REPO / "COMMIT.txt").read_text().strip()
      if (REPO / "COMMIT.txt").exists() else "KHÔNG có COMMIT.txt")


In [ ]:
cands = [p.parent for p in Path("/kaggle/input").glob("**/selected-contexts") if p.is_dir()]
assert cands, "Không thấy selected-contexts/ — chưa gắn dữ liệu thi"
DATA_DIR = cands[0]
CLEAN = Path("/kaggle/working/clean")
CLEAN.mkdir(exist_ok=True)

os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["TRAIN_JSON"] = str(DATA_DIR / "train.json")
os.environ["CLEAN_DIR"] = str(CLEAN)
os.environ["PYTHONPATH"] = "src"
print("DATA_DIR", DATA_DIR, "|", len(list((DATA_DIR / "selected-contexts").glob("*.json"))), "file (máy dev: 8.532)")


In [ ]:
# `!python` trong IPython KHÔNG ném lỗi khi mã thoát khác 0 — nó in traceback rồi cell
# vẫn xanh, và bước sau chạy trên dữ liệu thiếu. subprocess.run(check=True) mới dừng thật.
for s in ("scripts/clean_data.py", "scripts/build_chunks.py"):
    print("===", s, flush=True)
    subprocess.run([sys.executable, s], check=True)


In [ ]:
# bm25.pkl KHÔNG do build_chunks.py sinh (xem docstring `build_sft.nap_index`) —
# nó là cache của `eval_retrieval.load_index`. `infer_submit.ipynb` assert file này
# có mặt, nên dựng ngay ở đây (~50s) thay vì để infer tự dựng trên kernel có GPU.
import json, time
from legalqa.bm25 import ChunkIndex
from legalqa.chunking import Chunk

t0 = time.time()
chunks = []
with (CLEAN / "chunks.jsonl").open(encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        chunks.append(Chunk(**{k: d[k] for k in ("chunk_id", "doc_id", "start", "end", "text")}))
idx = ChunkIndex()
idx.build(chunks)
idx.save(CLEAN / "bm25.pkl")
print(f"{len(chunks):,} Chunk (máy dev: 287.358) | bm25 {time.time()-t0:.0f}s")


In [ ]:
# Ba file này là hợp đồng với `infer_submit.ipynb` cell 3. Kiểm ở ĐÂY, nơi sửa còn rẻ,
# chứ không để infer chết sau khi đã tải xong reranker trên kernel có GPU.
for n in ("chunks.jsonl", "corpus.jsonl", "bm25.pkl"):
    p = CLEAN / n
    assert p.exists() and p.stat().st_size > 0, f"thiếu {n}"
    print(f"{n:16s} {p.stat().st_size:>13,} B")
shutil.rmtree(REPO)   # 30 MB codebase không thuộc về output clean/
